In [5]:
# Import necessary libraries
import pandas as pd
import logging
from m5_forecasting.preprocessing.data_processor import DataProcessor  
from m5_forecasting.config import Config

# Set up logging to display outputs in the notebook
logging.basicConfig(level=logging.INFO)


config = Config.from_yaml("configs/project_config.yml")

# Load the CSVs
calendar = pd.read_csv('data/calendar.csv')
sell_prices = pd.read_csv('data/sell_prices.csv')
sales_data = pd.read_csv('data/sales_train_evaluation.csv')

# Initialize DataProcessor with the loaded data
data_processor = DataProcessor(config, sales_data, calendar, sell_prices)

# Run preprocess_data to see the outputs
sales_df, calendar_df, sell_price_df, prod_info_df = data_processor.preprocess_data()

# Display the results
display(sales_df.head())
display(calendar_df.head())
display(sell_price_df.head())
display(prod_info_df.head())


,unique_id,item_id,dept_id,cat_id,store_id,state_id,ds_id,y,ds,wm_yr_wk,release
0,HOBBIES_1_008_CA_1,HOBBIES_1_008,HOBBIES_1,HOBBIES,CA_1,CA,d_1,12,2011-01-29,11101,11101
1,HOBBIES_1_009_CA_1,HOBBIES_1_009,HOBBIES_1,HOBBIES,CA_1,CA,d_1,2,2011-01-29,11101,11101
2,HOBBIES_1_010_CA_1,HOBBIES_1_010,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,11101
3,HOBBIES_1_012_CA_1,HOBBIES_1_012,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,11101
4,HOBBIES_1_015_CA_1,HOBBIES_1_015,HOBBIES_1,HOBBIES,CA_1,CA,d_1,4,2011-01-29,11101,11101


,ds_id,ds,day_of_week,is_weekend,day_of_month,week_of_month,month,week_num_year,year,num_events,wm_yr_wk
0,d_1,2011-01-29,5,1,29,5,1,4,2011,0,11101
1,d_2,2011-01-30,6,1,30,5,1,4,2011,0,11101
2,d_3,2011-01-31,0,0,31,5,1,5,2011,0,11101
3,d_4,2011-02-01,1,0,1,1,2,5,2011,0,11101
4,d_5,2011-02-02,2,0,2,1,2,5,2011,0,11101


,unique_id,ds,sell_price
0,HOBBIES_1_001_CA_1,2013-07-13,9.58
1,HOBBIES_1_001_CA_1,2013-07-14,9.58
2,HOBBIES_1_001_CA_1,2013-07-15,9.58
3,HOBBIES_1_001_CA_1,2013-07-16,9.58
4,HOBBIES_1_001_CA_1,2013-07-17,9.58


,unique_id,item_id,dept_id,cat_id,store_id,state_id
0,HOBBIES_1_001_CA_1,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA
1,HOBBIES_1_002_CA_1,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA
2,HOBBIES_1_003_CA_1,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA
3,HOBBIES_1_004_CA_1,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA
4,HOBBIES_1_005_CA_1,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA


In [6]:
# --- Validation Checks ---

# Sales_df Checks
print("\n--- sales_df Checks ---")
# Check for duplicates in 'unique_id' and 'ds'
duplicates_sales = sales_df[sales_df.duplicated(subset=["unique_id", "ds"], keep=False)]
if duplicates_sales.empty:
    print("No duplicates in 'unique_id' and 'ds' columns of sales_df.")
else:
    print("Duplicates found in 'unique_id' and 'ds' columns of sales_df:", duplicates_sales)

# Ensure no NULL values in the 'y' column of sales_df
null_y_values = sales_df[sales_df["y"].isnull()]
if null_y_values.empty:
    print("No NULL values in 'y' column of sales_df.")
else:
    print("NULL values found in 'y' column of sales_df:", null_y_values)

# Ensure 'unique_id' and 'ds' have no NULL values in sales_df
null_unique_id_ds = sales_df[sales_df["unique_id"].isnull() | sales_df["ds"].isnull()]
if null_unique_id_ds.empty:
    print("No NULL values in 'unique_id' or 'ds' columns of sales_df.")
else:
    print("NULL values found in 'unique_id' or 'ds' columns of sales_df:", null_unique_id_ds)

# Ensure 'y' values are within expected range (e.g., non-negative)
if (sales_df["y"] < 0).any():
    print("Negative values found in 'y' column of sales_df.")
else:
    print("All 'y' values in sales_df are non-negative.")

# Calendar_df Checks
print("\n--- calendar_df Checks ---")
# Check that 'ds' is unique in calendar_df
if calendar_df["ds"].is_unique:
    print("'ds' column in calendar_df is unique.")
else:
    print("Duplicates found in 'ds' column of calendar_df.")

# Verify continuous date range without missing dates in calendar_df
expected_dates = pd.date_range(start=calendar_df["ds"].min(), end=calendar_df["ds"].max(), freq="D")
missing_dates = expected_dates.difference(calendar_df["ds"])
if missing_dates.empty:
    print("No missing dates in 'ds' column of calendar_df.")
else:
    print("Missing dates in 'ds' column of calendar_df:", missing_dates)

# Ensure no NULL values in the calendar_df
if calendar_df.isnull().sum().sum() == 0:
    print("No NULL values in calendar_df.")
else:
    print("NULL values found in calendar_df.")

# Sell_price_df Checks
print("\n--- sell_price_df Checks ---")
# Check for duplicates in 'unique_id' and 'ds' in sell_price_df
duplicates_sell_price = sell_price_df[sell_price_df.duplicated(subset=["unique_id", "ds"], keep=False)]
if duplicates_sell_price.empty:
    print("No duplicates in 'unique_id' and 'ds' columns of sell_price_df.")
else:
    print("Duplicates found in 'unique_id' and 'ds' columns of sell_price_df:", duplicates_sell_price)

# Ensure no NULL values in 'sell_price' or 'ds' columns in sell_price_df
null_sell_price_ds = sell_price_df[sell_price_df["sell_price"].isnull() | sell_price_df["ds"].isnull()]
if null_sell_price_ds.empty:
    print("No NULL values in 'sell_price' or 'ds' columns of sell_price_df.")
else:
    print("NULL values found in 'sell_price' or 'ds' columns of sell_price_df:", null_sell_price_ds)

# Ensure 'sell_price' values are within a valid range (e.g., non-negative)
if (sell_price_df["sell_price"] < 0).any():
    print("Negative values found in 'sell_price' column of sell_price_df.")
else:
    print("All 'sell_price' values in sell_price_df are non-negative.")

# Prod_info_df Checks
print("\n--- prod_info_df Checks ---")
# Check that 'unique_id' is unique in prod_info_df
if prod_info_df["unique_id"].is_unique:
    print("'unique_id' column in prod_info_df is unique.")
else:
    print("Duplicates found in 'unique_id' column of prod_info_df.")

# Ensure no NULL values in prod_info_df
if prod_info_df.isnull().sum().sum() == 0:
    print("No NULL values in prod_info_df.")
else:
    print("NULL values found in prod_info_df.")

print("\n--- Validation Complete ---")



--- sales_df Checks ---
No duplicates in 'unique_id' and 'ds' columns of sales_df.
No NULL values in 'y' column of sales_df.
No NULL values in 'unique_id' or 'ds' columns of sales_df.
All 'y' values in sales_df are non-negative.

--- calendar_df Checks ---
'ds' column in calendar_df is unique.
No missing dates in 'ds' column of calendar_df.
No NULL values in calendar_df.

--- sell_price_df Checks ---
No duplicates in 'unique_id' and 'ds' columns of sell_price_df.
No NULL values in 'sell_price' or 'ds' columns of sell_price_df.
All 'sell_price' values in sell_price_df are non-negative.

--- prod_info_df Checks ---
'unique_id' column in prod_info_df is unique.
No NULL values in prod_info_df.

--- Validation Complete ---


In [14]:
# Step 1: Count distinct unique IDs
distinct_unique_ids = sales_df['unique_id'].nunique()
print(f"Number of distinct unique IDs: {distinct_unique_ids}")

# Step 2: Calculate the length of each unique_id's time series
unique_id_lengths = sales_df.groupby('unique_id')['ds'].nunique().reset_index()
unique_id_lengths.columns = ['unique_id', 'ds_length']

# Step 3: Check if all unique_ids have the same ds length
length_counts = unique_id_lengths['ds_length'].value_counts()

# Display results
print(f"Count of each unique time length:\n{length_counts}")

# Optional: Display a summary of unique_id lengths to understand if they are uniform or vary
if length_counts.nunique() == 1:
    print("All unique_ids have the same ds length.")
else:
    print("There are different ds lengths among unique_ids.")
    print(f"Distribution of ds lengths:\n{length_counts}")

# Find the shortest length and corresponding unique_id(s)
shortest_length = unique_id_lengths['ds_length'].min()
shortest_ids = unique_id_lengths[unique_id_lengths['ds_length'] == shortest_length]

# Display shortest length information with min and max dates
print(f"Shortest length of a time series: {shortest_length}")
for unique_id in shortest_ids['unique_id']:
    min_date = sales_df[sales_df['unique_id'] == unique_id]['ds'].min()
    max_date = sales_df[sales_df['unique_id'] == unique_id]['ds'].max()
    print(f"Unique ID: {unique_id}, Min Date: {min_date}, Max Date: {max_date}")

# Find the largest length and corresponding unique_id(s)
largest_length = unique_id_lengths['ds_length'].max()
largest_ids = unique_id_lengths[unique_id_lengths['ds_length'] == largest_length]

# Display largest length information with min and max dates
print(f"Largest length of a time series: {largest_length}")


Number of distinct unique IDs: 30490
Count of each unique time length:
1941    10932
1934     1043
1927      544
1906      301
1920      280
        ...  
135         1
289         1
177         1
191         1
331         1
Name: ds_length, Length: 259, dtype: int64
There are different ds lengths among unique_ids.
Distribution of ds lengths:
1941    10932
1934     1043
1927      544
1906      301
1920      280
        ...  
135         1
289         1
177         1
191         1
331         1
Name: ds_length, Length: 259, dtype: int64
Shortest length of a time series: 100
Unique ID: FOODS_3_595_CA_1, Min Date: 2016-02-13 00:00:00, Max Date: 2016-05-22 00:00:00
Unique ID: FOODS_3_595_CA_3, Min Date: 2016-02-13 00:00:00, Max Date: 2016-05-22 00:00:00
Unique ID: HOUSEHOLD_1_020_WI_2, Min Date: 2016-02-13 00:00:00, Max Date: 2016-05-22 00:00:00
Unique ID: HOUSEHOLD_1_278_CA_3, Min Date: 2016-02-13 00:00:00, Max Date: 2016-05-22 00:00:00
Unique ID: HOUSEHOLD_1_311_CA_2, Min Date: 2016-02-1